In [255]:
import os
import pandas as pd
import numpy as np
import matplotlib
import sklearn
print(os.getcwd())

/home/michele/big-project-FOR


# Reading CSV and creating pairs


In [256]:
edificio1= pd.read_csv('Edificio1.csv')
edificio2= pd.read_csv('Edificio2.csv')

edificio1_pairs= edificio1.rename_axis('index1').reset_index().merge(edificio1.rename_axis('index2').reset_index(), how='cross')
edificio2_pairs= edificio2.rename_axis('index1').reset_index().merge(edificio2.rename_axis('index2').reset_index(), how='cross')

In [257]:
first_point_cols= ['x_x','y_x','z_x']
second_point_cols=['x_y','y_y','z_y']
diff_cols= ['dx','dy','dz']

# Computing distances 

In [258]:
def process_edificio(dataframe):
    #distances and deltas on coordinates
    dataframe[diff_cols] = (dataframe[second_point_cols].values - dataframe[first_point_cols].values)
    dataframe['euclidean_distance'] = np.sqrt((dataframe[diff_cols]**2).sum(axis=1)) #axis=1 sum on columns 
    
    
    # conditions
    condition_near = (dataframe['euclidean_distance'] <= 4.0)
    
    
    condition_tunnel = (
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dy'].abs() <= 0.5)) | 
        ((dataframe['dy'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5)) | 
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5))
    )
    condition_tunnel_and_not_far = (dataframe['euclidean_distance'] <= 11.0) & condition_tunnel
    
    # final connection condition
    dataframe["are_connected"] = condition_near | condition_tunnel_and_not_far
    return dataframe



In [259]:
edificio1_pairs = process_edificio(edificio1_pairs)
edificio2_pairs = process_edificio(edificio2_pairs)

edificio1_pairs.head()
print("\nSHAPE", edificio1_pairs.shape)



SHAPE (4096, 13)


In [260]:
edificio1_pairs["d_horizontal"] = np.sqrt(edificio1_pairs['dx']**2 + edificio1_pairs['dy']**2)
edificio1_pairs["d_vertical"] = edificio1_pairs['dz']
edificio2_pairs["d_horizontal"] = np.sqrt(edificio2_pairs['dx']**2 + edificio2_pairs['dy']**2)
edificio2_pairs["d_vertical"] = edificio2_pairs['dz']

edificio1_pairs.drop(columns=['dx','dy','dz'], inplace=True)
edificio2_pairs.drop(columns=['dx','dy','dz'], inplace=True)

edificio1_pairs.head(10)

,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,are_connected,d_horizontal,d_vertical
0,0,2.5,3.25,0.0,0,2.5,3.25,0.0,0.000000,True,0.000000,0.0
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,True,8.500000,0.0
2,0,2.5,3.25,0.0,2,-6.0,11.75,0.0,12.020815,False,12.020815,0.0
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,True,8.500000,0.0
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,True,0.000000,11.0
5,0,2.5,3.25,0.0,5,-6.0,3.25,11.0,13.901439,False,8.500000,11.0
6,0,2.5,3.25,0.0,6,-6.0,11.75,11.0,16.294171,False,12.020815,11.0
7,0,2.5,3.25,0.0,7,2.5,11.75,11.0,13.901439,False,8.500000,11.0
8,0,2.5,3.25,0.0,8,-3.0,5.00,11.0,12.422258,False,5.771698,11.0
9,0,2.5,3.25,0.0,9,-3.0,10.00,11.0,14.028988,False,8.707037,11.0


In [261]:
ASCENDING_SPEED = 1.0
DESCENDING_SPEED = 2.0
HORIZONTAL_SPEED = 1.5
#oblique movements with lateral a and vertical b is computed as max(a/speed_horizontal, b/speed_vertical)

ASCENDING_ENERGY= 50
DESCENDING_ENERGY= 5
HORIZONTAL_ENERGY = 10

In [262]:
edificio1_pairs=edificio1_pairs[edificio1_pairs['are_connected']==True]
edificio2_pairs=edificio2_pairs[edificio2_pairs['are_connected']==True]
edificio1_pairs.head(10)

,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,are_connected,d_horizontal,d_vertical
0,0,2.5,3.25,0.0,0,2.5,3.25,0.0,0.000000,True,0.000000,0.0
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,True,8.500000,0.0
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,True,8.500000,0.0
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,True,0.000000,11.0
18,0,2.5,3.25,0.0,18,-1.5,3.50,0.0,4.007805,True,4.007805,0.0
19,0,2.5,3.25,0.0,19,1.5,3.50,0.0,1.030776,True,1.030776,0.0
64,1,-6.0,3.25,0.0,0,2.5,3.25,0.0,8.500000,True,8.500000,0.0
65,1,-6.0,3.25,0.0,1,-6.0,3.25,0.0,0.000000,True,0.000000,0.0
66,1,-6.0,3.25,0.0,2,-6.0,11.75,0.0,8.500000,True,8.500000,0.0
69,1,-6.0,3.25,0.0,5,-6.0,3.25,11.0,11.000000,True,0.000000,11.0


In [263]:
def compute_time_travel(dataframe):
    d_horizontal = dataframe['d_horizontal']
    d_vertical = dataframe['d_vertical']
    rows_horizontal = (d_vertical == 0)
    rows_ascending = (d_horizontal == 0) & (d_vertical > 0)
    rows_descending = (d_horizontal == 0) & (d_vertical < 0)
    rows_oblique_ascending = (d_horizontal != 0) & (d_vertical > 0)
    rows_oblique_descending = (d_horizontal != 0) & (d_vertical < 0)
    dataframe.loc[rows_horizontal, 'time_travel'] = dataframe.loc[rows_horizontal, 'd_horizontal'] / HORIZONTAL_SPEED
    dataframe.loc[rows_ascending, 'time_travel'] = dataframe.loc[rows_ascending, 'd_vertical'] / ASCENDING_SPEED
    dataframe.loc[rows_descending, 'time_travel'] = -dataframe.loc[rows_descending, 'd_vertical'] / DESCENDING_SPEED

    dataframe.loc[rows_oblique_ascending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_ascending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     dataframe.loc[rows_oblique_ascending, 'd_vertical'] / ASCENDING_SPEED)
    
    dataframe.loc[rows_oblique_descending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_descending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     -dataframe.loc[rows_oblique_descending, 'd_vertical'] / DESCENDING_SPEED)
    return dataframe

edificio1_pairs = compute_time_travel(edificio1_pairs)
edificio2_pairs = compute_time_travel(edificio2_pairs)

edificio1_pairs.head(10)


,index1,x_x,y_x,z_x,index2,x_y,y_y,z_y,euclidean_distance,are_connected,d_horizontal,d_vertical,time_travel
0,0,2.5,3.25,0.0,0,2.5,3.25,0.0,0.000000,True,0.000000,0.0,0.000000
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,8.500000,True,8.500000,0.0,5.666667
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,8.500000,True,8.500000,0.0,5.666667
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,11.000000,True,0.000000,11.0,11.000000
18,0,2.5,3.25,0.0,18,-1.5,3.50,0.0,4.007805,True,4.007805,0.0,2.671870
19,0,2.5,3.25,0.0,19,1.5,3.50,0.0,1.030776,True,1.030776,0.0,0.687184
64,1,-6.0,3.25,0.0,0,2.5,3.25,0.0,8.500000,True,8.500000,0.0,5.666667
65,1,-6.0,3.25,0.0,1,-6.0,3.25,0.0,0.000000,True,0.000000,0.0,0.000000
66,1,-6.0,3.25,0.0,2,-6.0,11.75,0.0,8.500000,True,8.500000,0.0,5.666667
69,1,-6.0,3.25,0.0,5,-6.0,3.25,11.0,11.000000,True,0.000000,11.0,11.000000
